In [1]:
# -*- coding: utf-8 -*-

import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import classification_report, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# =========================
# 1. 固定随机种子
# =========================

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# =========================
# 2. 模拟医保主单数据
# =========================

def generate_main_data(n=3000):
    disease_types = ["肺炎", "高血压", "糖尿病", "腰椎病", "冠心病", "脑梗死", "胃炎"]
    hospital_levels = ["一级", "二级", "三级"]
    departments = ["内科", "外科", "康复科", "心内科", "神经内科", "消化科"]

    rows = []

    for i in range(n):
        claim_id = f"C{i:06d}"
        patient_id = f"P{random.randint(1, 1200):05d}"
        hospital_id = f"H{random.randint(1, 50):03d}"

        disease = random.choice(disease_types)
        hospital_level = random.choices(
            hospital_levels,
            weights=[0.25, 0.45, 0.30]
        )[0]
        dept = random.choice(departments)

        age = random.randint(18, 90)
        inpatient_days = max(1, int(np.random.normal(8, 4)))
        visit_cnt_30d = np.random.poisson(1.5)

        rows.append({
            "claim_id": claim_id,
            "patient_id": patient_id,
            "hospital_id": hospital_id,
            "disease_type": disease,
            "hospital_level": hospital_level,
            "department": dept,
            "age": age,
            "inpatient_days": inpatient_days,
            "visit_cnt_30d": visit_cnt_30d
        })

    return pd.DataFrame(rows)


main_df = generate_main_data(3000)


# =========================
# 3. 模拟医保明细数据
# =========================

def generate_detail_data(main_df):
    detail_rows = []

    item_categories = ["drug", "exam", "treatment", "rehab", "material"]

    for _, row in main_df.iterrows():
        claim_id = row["claim_id"]
        disease = row["disease_type"]
        dept = row["department"]

        # 每张主单生成 5 到 25 条费用明细
        detail_count = random.randint(5, 25)

        # 不同病种/科室的费用结构略有差异
        if disease in ["高血压", "糖尿病", "冠心病"]:
            category_weights = [0.50, 0.20, 0.15, 0.05, 0.10]
        elif disease in ["腰椎病"] or dept == "康复科":
            category_weights = [0.25, 0.20, 0.20, 0.25, 0.10]
        elif disease in ["脑梗死"]:
            category_weights = [0.35, 0.25, 0.20, 0.10, 0.10]
        else:
            category_weights = [0.35, 0.30, 0.20, 0.05, 0.10]

        for j in range(detail_count):
            category = random.choices(item_categories, weights=category_weights)[0]

            if category == "drug":
                amount = np.random.gamma(2, 120)
            elif category == "exam":
                amount = np.random.gamma(2, 180)
            elif category == "treatment":
                amount = np.random.gamma(2, 150)
            elif category == "rehab":
                amount = np.random.gamma(2, 100)
            else:
                amount = np.random.gamma(2, 200)

            detail_rows.append({
                "claim_id": claim_id,
                "item_id": f"I{random.randint(1, 500):04d}",
                "item_category": category,
                "amount": round(float(amount), 2)
            })

    return pd.DataFrame(detail_rows)


detail_df = generate_detail_data(main_df)


# =========================
# 4. 模拟出院小结文本
# =========================

def generate_discharge_summary(row):
    disease = row["disease_type"]
    dept = row["department"]
    visit_cnt = row["visit_cnt_30d"]
    days = row["inpatient_days"]

    base_text = {
        "肺炎": "患者因咳嗽发热入院，完善血常规胸部影像检查，给予抗感染对症治疗，病情好转后出院。",
        "高血压": "患者因血压控制不佳入院，调整降压药物方案，监测血压变化，建议规律服药定期复诊。",
        "糖尿病": "患者因血糖控制不佳入院，完善血糖监测，调整胰岛素及降糖药物方案，进行饮食运动指导。",
        "腰椎病": "患者因腰腿疼痛入院，给予理疗针灸牵引及止痛治疗，症状缓解后出院。",
        "冠心病": "患者因胸闷胸痛入院，完善心电图心肌酶检查，给予抗血小板调脂改善循环治疗。",
        "脑梗死": "患者因肢体乏力入院，完善头颅影像检查，给予抗血小板改善循环及康复训练。",
        "胃炎": "患者因腹痛腹胀入院，完善胃镜及相关检查，给予抑酸护胃对症治疗。"
    }

    text = base_text[disease]

    # 加入一些可能与风险相关的文本模式
    if visit_cnt >= 4:
        text += " 患者近期多次就诊，反复开药及复查。"

    if dept == "康复科" or disease in ["腰椎病", "脑梗死"]:
        text += " 住院期间多次进行康复治疗、理疗、针灸和功能训练。"

    if days <= 2:
        text += " 住院时间较短，检查项目较多，需结合费用明细进一步核查。"

    if random.random() < 0.08:
        text += " 部分治疗项目记录较集中，费用结构存在一定异常。"

    return text


main_df["discharge_summary"] = main_df.apply(generate_discharge_summary, axis=1)


# =========================
# 5. 明细表汇总成主单级特征
# =========================

def aggregate_detail_features(detail_df):
    total_fee = detail_df.groupby("claim_id")["amount"].sum().rename("total_fee")
    detail_cnt = detail_df.groupby("claim_id")["item_id"].count().rename("detail_cnt")

    category_fee = (
        detail_df
        .pivot_table(
            index="claim_id",
            columns="item_category",
            values="amount",
            aggfunc="sum",
            fill_value=0
        )
    )

    category_fee.columns = [f"{c}_fee" for c in category_fee.columns]

    agg = pd.concat([total_fee, detail_cnt, category_fee], axis=1).reset_index()

    for col in ["drug_fee", "exam_fee", "treatment_fee", "rehab_fee", "material_fee"]:
        if col not in agg.columns:
            agg[col] = 0.0

    agg["drug_ratio"] = agg["drug_fee"] / agg["total_fee"]
    agg["exam_ratio"] = agg["exam_fee"] / agg["total_fee"]
    agg["treatment_ratio"] = agg["treatment_fee"] / agg["total_fee"]
    agg["rehab_ratio"] = agg["rehab_fee"] / agg["total_fee"]
    agg["material_ratio"] = agg["material_fee"] / agg["total_fee"]

    return agg


detail_agg_df = aggregate_detail_features(detail_df)

data_df = main_df.merge(detail_agg_df, on="claim_id", how="left")


# =========================
# 6. 模拟标签：是否高风险/疑似违规
# =========================
# 注意：
# 真实项目中 label 应来自历史审核结果、人工稽核结果、行政处罚结果等。
# 这里为了演示，用规则 + 随机噪声生成标签。

def generate_label(row):
    risk_score = 0

    if row["visit_cnt_30d"] >= 4:
        risk_score += 2

    if row["drug_ratio"] >= 0.65:
        risk_score += 1

    if row["exam_ratio"] >= 0.45:
        risk_score += 1

    if row["rehab_ratio"] >= 0.30:
        risk_score += 1

    if row["total_fee"] >= data_df["total_fee"].quantile(0.85):
        risk_score += 1

    if row["inpatient_days"] <= 2 and row["total_fee"] >= data_df["total_fee"].quantile(0.60):
        risk_score += 2

    if "异常" in row["discharge_summary"] or "进一步核查" in row["discharge_summary"]:
        risk_score += 1

    # 加一点随机性，模拟真实世界里标签不是完全由规则决定
    risk_score += np.random.binomial(1, 0.05)

    return 1 if risk_score >= 3 else 0


data_df["label"] = data_df.apply(generate_label, axis=1)

print("样本数量：", len(data_df))
print("高风险样本比例：")
print(data_df["label"].value_counts(normalize=True))


# =========================
# 7. 切分训练集和测试集
# =========================

train_df, test_df = train_test_split(
    data_df,
    test_size=0.2,
    random_state=42,
    stratify=data_df["label"]
)


# =========================
# 8. LDA：把出院小结变成主题特征
# =========================
# 说明：
# 这里为了方便演示，中文没有做 jieba 分词，
# 因为模拟文本中词语相对固定。
# 真实项目建议先做中文分词，再做 LDA。

vectorizer = CountVectorizer(max_features=300)
train_text_count = vectorizer.fit_transform(train_df["discharge_summary"])
test_text_count = vectorizer.transform(test_df["discharge_summary"])

lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="batch"
)

train_topic = lda.fit_transform(train_text_count)
test_topic = lda.transform(test_text_count)

topic_cols = [f"topic_{i}" for i in range(train_topic.shape[1])]

train_topic_df = pd.DataFrame(train_topic, columns=topic_cols, index=train_df.index)
test_topic_df = pd.DataFrame(test_topic, columns=topic_cols, index=test_df.index)


# 查看每个主题的关键词
feature_names = np.array(vectorizer.get_feature_names_out())

print("\nLDA 学到的主题关键词：")
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[-8:][::-1]
    top_words = feature_names[top_words_idx]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")


# =========================
# 9. 准备结构化特征
# =========================

numeric_cols = [
    "age",
    "inpatient_days",
    "visit_cnt_30d",
    "total_fee",
    "detail_cnt",
    "drug_fee",
    "exam_fee",
    "treatment_fee",
    "rehab_fee",
    "material_fee",
    "drug_ratio",
    "exam_ratio",
    "treatment_ratio",
    "rehab_ratio",
    "material_ratio"
]

categorical_cols = [
    "disease_type",
    "hospital_level",
    "department"
]

scaler = StandardScaler()
train_num = scaler.fit_transform(train_df[numeric_cols])
test_num = scaler.transform(test_df[numeric_cols])

# 兼容不同 sklearn 版本
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

train_cat = encoder.fit_transform(train_df[categorical_cols])
test_cat = encoder.transform(test_df[categorical_cols])


# 拼接：数值特征 + 类别特征 + LDA主题特征
X_train = np.hstack([
    train_num,
    train_cat,
    train_topic_df.values
]).astype(np.float32)

X_test = np.hstack([
    test_num,
    test_cat,
    test_topic_df.values
]).astype(np.float32)

y_train = train_df["label"].values.astype(np.float32)
y_test = test_df["label"].values.astype(np.float32)

print("\n训练特征维度：", X_train.shape)
print("测试特征维度：", X_test.shape)


# =========================
# 10. 构造 PyTorch Dataset
# =========================

class MedicalClaimDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = MedicalClaimDataset(X_train, y_train)
test_dataset = MedicalClaimDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)


# =========================
# 11. 定义 DNN 模型
# =========================

class DNNFraudModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


input_dim = X_train.shape[1]
model = DNNFraudModel(input_dim)

# BCEWithLogitsLoss = Sigmoid + 二分类交叉熵
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# =========================
# 12. 训练 DNN
# =========================

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        logits = model(batch_X)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)

    avg_loss = total_loss / len(train_dataset)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


# =========================
# 13. 测试集评估
# =========================

model.eval()
all_probs = []
all_preds = []

with torch.no_grad():
    for batch_X, _ in test_loader:
        logits = model(batch_X)
        probs = torch.sigmoid(logits).numpy().reshape(-1)
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)

print("\n分类评估结果：")
print(classification_report(y_test, all_preds, digits=4))

auc = roc_auc_score(y_test, all_probs)
print("AUC:", round(auc, 4))


# =========================
# 14. 模拟一条新医保单据并预测
# =========================

new_main = pd.DataFrame([{
    "claim_id": "NEW001",
    "patient_id": "P99999",
    "hospital_id": "H001",
    "disease_type": "腰椎病",
    "hospital_level": "二级",
    "department": "康复科",
    "age": 67,
    "inpatient_days": 2,
    "visit_cnt_30d": 5,
    "discharge_summary": "患者因腰腿疼痛入院，住院期间多次进行康复治疗、理疗、针灸和牵引治疗。近期多次就诊，住院时间较短，检查项目较多，需结合费用明细进一步核查。"
}])

new_detail = pd.DataFrame([
    {"claim_id": "NEW001", "item_id": "I0001", "item_category": "rehab", "amount": 1200},
    {"claim_id": "NEW001", "item_id": "I0002", "item_category": "rehab", "amount": 900},
    {"claim_id": "NEW001", "item_id": "I0003", "item_category": "exam", "amount": 1500},
    {"claim_id": "NEW001", "item_id": "I0004", "item_category": "drug", "amount": 300},
    {"claim_id": "NEW001", "item_id": "I0005", "item_category": "material", "amount": 200},
])

new_detail_agg = aggregate_detail_features(new_detail)
new_data = new_main.merge(new_detail_agg, on="claim_id", how="left")

# LDA 主题特征
new_text_count = vectorizer.transform(new_data["discharge_summary"])
new_topic = lda.transform(new_text_count)

# 数值特征
new_num = scaler.transform(new_data[numeric_cols])

# 类别特征
new_cat = encoder.transform(new_data[categorical_cols])

# 拼接
X_new = np.hstack([
    new_num,
    new_cat,
    new_topic
]).astype(np.float32)

model.eval()
with torch.no_grad():
    new_logits = model(torch.tensor(X_new, dtype=torch.float32))
    new_prob = torch.sigmoid(new_logits).item()

print("\n新单据预测结果：")
print("高风险概率：", round(new_prob, 4))
print("预测结果：", "高风险/疑似违规" if new_prob >= 0.5 else "低风险/正常")

样本数量： 3000
高风险样本比例：
label
0    0.930667
1    0.069333
Name: proportion, dtype: float64

LDA 学到的主题关键词：
Topic 0: 患者因胸闷胸痛入院, 给予抗血小板调脂改善循环治疗, 完善心电图心肌酶检查, 需结合费用明细进一步核查, 检查项目较多, 住院时间较短, 费用结构存在一定异常, 部分治疗项目记录较集中
Topic 1: 监测血压变化, 建议规律服药定期复诊, 调整降压药物方案, 患者因血压控制不佳入院, 完善血糖监测, 患者因血糖控制不佳入院, 调整胰岛素及降糖药物方案, 进行饮食运动指导
Topic 2: 患者因腹痛腹胀入院, 给予抑酸护胃对症治疗, 完善胃镜及相关检查, 反复开药及复查, 患者近期多次就诊, 需结合费用明细进一步核查, 检查项目较多, 住院时间较短
Topic 3: 给予抗感染对症治疗, 病情好转后出院, 患者因咳嗽发热入院, 完善血常规胸部影像检查, 费用结构存在一定异常, 部分治疗项目记录较集中, 反复开药及复查, 患者近期多次就诊
Topic 4: 针灸和功能训练, 住院期间多次进行康复治疗, 理疗, 患者因腰腿疼痛入院, 症状缓解后出院, 给予理疗针灸牵引及止痛治疗, 患者因肢体乏力入院, 完善头颅影像检查

训练特征维度： (2400, 36)
测试特征维度： (600, 36)
Epoch [5/20], Loss: 0.1805
Epoch [10/20], Loss: 0.1494
Epoch [15/20], Loss: 0.1289
Epoch [20/20], Loss: 0.1115

分类评估结果：
              precision    recall  f1-score   support

         0.0     0.9665    0.9839    0.9751       558
         1.0     0.7188    0.5476    0.6216        42

    accuracy                         0.9533       600
   macro avg     0.8426    0.7657    0.7984   